In [1]:
import os  # Lets us set the Java location before Spark starts.
import shutil  # Finds Ubuntu's Java command.
from pathlib import Path  # Helps us build filesystem paths.
from pyspark.sql import SparkSession  # Creates our Spark session.
from delta import configure_spark_with_delta_pip  # Adds Delta's Java package.

java_command = shutil.which("java")  # Find Java in the WSL environment.
if java_command is None:  # Check that Java was found.
    raise RuntimeError("Java was not found in WSL")  # Stop with a clear message if it was not.

java_home = Path(java_command).resolve().parent.parent  # Find the Java installation folder.
os.environ["JAVA_HOME"] = str(java_home)  # Point Spark to Ubuntu's Java.

project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Locate our project.

builder = (  # Define Spark's settings before it starts.
    SparkSession.builder
    .master("local[2]")  # Use two processing threads on this computer.
    .appName("RetailSilverCustomers")  # Name this Silver job.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta features.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")  # Enable Delta tables.
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start Spark with Delta support.

print("Project found:", project_root.is_dir())  # Confirm the project path works.
print("Spark version:", spark.version)  # Confirm Spark started.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 18:10:43 WARN Utils: Your hostname, USMAN-ALI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/09 18:10:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/usmanali611b/.venvs/retail-lakehouse/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/usmanali611b/.ivy2.5.2/cache
The jars for the packages stored in: /home/usmanali611b/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0472b054-906c-4b8a-b595-993813e51bbb;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.1

Project found: True
Spark version: 4.1.1


In [2]:
bronze_customers_path = project_root / "lakehouse" / "bronze" / "data" / "customers"  # Locate the saved Bronze table.
customers_bronze_df = spark.read.format("delta").load(str(bronze_customers_path))  # Read its current Delta version.

print("Bronze customer rows:", customers_bronze_df.count())  # Count the records available for Silver.
customers_bronze_df.printSchema()  # Display column names and Spark data types.

customers_bronze_df.select(  # Choose useful columns for a small preview.
    "customer_id", "city", "loyalty_tier", "join_date", "active_flag"  # Include ID, categories, date, and flag.
).show(5, truncate=False)  # Display five rows without shortening the values.

26/10/09 18:12:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Bronze customer rows: 750
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- join_date: date (nullable = true)
 |-- active_flag: boolean (nullable = true)



[Stage 13:>                                                         (0 + 1) / 1]

+-----------+-------+------------+----------+-----------+
|customer_id|city   |loyalty_tier|join_date |active_flag|
+-----------+-------+------------+----------+-----------+
|C00001     |Lahore |Gold        |2026-05-13|true       |
|C00002     |Karachi|Silver      |2025-08-18|true       |
|C00003     |Karachi|Silver      |2025-04-19|true       |
|C00004     |Karachi|Bronze      |2024-02-03|true       |
|C00005     |Multan |Silver      |2025-03-25|true       |
+-----------+-------+------------+----------+-----------+
only showing top 5 rows


In [3]:
from pyspark.sql import functions as F  # Import Spark functions for column checks.

missing_id_count = customers_bronze_df.filter(  # Select customers without a usable ID.
    F.col("customer_id").isNull()  # Find database null values.
    | (F.trim(F.col("customer_id")) == "")  # Find empty IDs or IDs containing only spaces.
).count()  # Count those rows.

duplicate_id_groups = (  # Count customer IDs that appear more than once.
    customers_bronze_df
    .groupBy("customer_id")  # Put rows with the same ID together.
    .count()  # Count rows in each ID group.
    .filter(F.col("count") > 1)  # Keep IDs found in multiple rows.
    .count()  # Count those duplicate ID groups.
)

print("Missing or blank customer IDs:", missing_id_count)  # Show the first quality result.
print("Customer IDs appearing more than once:", duplicate_id_groups)  # Show the second result.

Missing or blank customer IDs: 0
Customer IDs appearing more than once: 0


In [4]:
required_text_columns = (  # List required fields stored as text.
    "first_name",  # Customer's first name.
    "last_name",  # Customer's last name.
    "email",  # Customer's email address.
    "loyalty_tier",  # Customer's loyalty category.
)

for column_name in required_text_columns:  # Check each required text field.
    missing_count = customers_bronze_df.filter(  # Select rows without a usable value.
        F.col(column_name).isNull()  # Find null values.
        | (F.trim(F.col(column_name)) == "")  # Find empty or spaces-only values.
    ).count()  # Count the matching rows.

    print(f"{column_name} missing or blank: {missing_count}")  # Show the result for this field.

for column_name in ("join_date", "active_flag"):  # Check the required date and boolean fields.
    missing_count = customers_bronze_df.filter(  # Select rows where the field is absent.
        F.col(column_name).isNull()  # A date or boolean is missing when it is null.
    ).count()  # Count those rows.

    print(f"{column_name} missing: {missing_count}")  # Show the result.

first_name missing or blank: 0
last_name missing or blank: 0
email missing or blank: 0
loyalty_tier missing or blank: 0
join_date missing: 0
active_flag missing: 0


In [5]:
allowed_tiers = ("Bronze", "Silver", "Gold")  # List the loyalty levels allowed by our source rules.

invalid_tier_count = customers_bronze_df.filter(  # Select customers with an unexpected tier.
    ~F.col("loyalty_tier").isin(*allowed_tiers)  # Keep values outside the allowed list.
).count()  # Count those customers.

invalid_email_count = customers_bronze_df.filter(  # Select customers failing our basic email rule.
    ~F.col("email").contains("@")  # Keep email values without an @ sign.
).count()  # Count those customers.

print("Invalid loyalty tiers:", invalid_tier_count)  # Display the tier check.
print("Emails without @:", invalid_email_count)  # Display the email check.

customers_bronze_df.groupBy("loyalty_tier").count().orderBy("loyalty_tier").show()  # Show how many customers are in each tier.

Invalid loyalty tiers: 0
Emails without @: 0
+------------+-----+
|loyalty_tier|count|
+------------+-----+
|      Bronze|  399|
|        Gold|   97|
|      Silver|  254|
+------------+-----+



In [6]:
phone_value = F.trim(F.col("phone"))  # Remove spaces at the ends of a phone value.
city_value = F.trim(F.col("city"))  # Remove spaces at the ends of a city value.

customers_clean_df = customers_bronze_df.select(  # Create a new DataFrame; leave Bronze unchanged.
    F.trim(F.col("customer_id")).alias("customer_id"),  # Clean the customer key.
    F.trim(F.col("first_name")).alias("first_name"),  # Clean the first name.
    F.trim(F.col("last_name")).alias("last_name"),  # Clean the last name.
    F.trim(F.col("email")).alias("email"),  # Clean email without changing its letter case.
    F.when(phone_value == "", F.lit(None)).otherwise(phone_value).alias("phone"),  # Treat a blank phone as missing.
    F.when(city_value == "", F.lit(None)).otherwise(city_value).alias("city"),  # Treat a blank city as missing.
    F.trim(F.col("loyalty_tier")).alias("loyalty_tier"),  # Clean the tier before checking it again.
    F.col("join_date"),  # Keep the date type received from PostgreSQL.
    F.col("active_flag"),  # Keep the boolean type received from PostgreSQL.
)

print("Cleaned customer rows:", customers_clean_df.count())  # Confirm cleaning did not remove rows.
customers_clean_df.show(3, truncate=False)  # Preview three cleaned records.

Cleaned customer rows: 750
+-----------+----------+---------+---------------------------+---------------+-------+------------+----------+-----------+
|customer_id|first_name|last_name|email                      |phone          |city   |loyalty_tier|join_date |active_flag|
+-----------+----------+---------+---------------------------+---------------+-------+------------+----------+-----------+
|C00001     |Maryam    |Iqbal    |maryam.iqbal.1@example.com |+92-333-8106245|Lahore |Gold        |2026-05-13|true       |
|C00002     |Sophia    |Davis    |sophia.davis.2@example.com |+92-336-9221514|Karachi|Silver      |2025-08-18|true       |
|C00003     |Fatima    |Wilson   |fatima.wilson.3@example.com|+92-318-8308766|Karachi|Silver      |2025-04-19|true       |
+-----------+----------+---------+---------------------------+---------------+-------+------------+----------+-----------+
only showing top 3 rows


In [7]:
from pyspark.sql.window import Window  # Lets us count rows sharing the same customer ID.

def missing_text(column_name):  # Build one reusable check for a required text field.
    column = F.col(column_name)  # Select the named Spark column.
    return column.isNull() | (column == "")  # Flag null or empty values after trimming.

customers_checked_df = customers_clean_df.withColumn(  # Add a count for each customer ID.
    "id_occurrences",  # Name the temporary count column.
    F.count(F.lit(1)).over(Window.partitionBy("customer_id")),  # Count rows with this same ID.
).withColumn(  # Add reasons why a row cannot enter Silver.
    "quality_issue",  # Store issue names together in one text column.
    F.concat_ws(  # Join issue names; null checks contribute nothing.
        "; ",  # Separate multiple issues with a semicolon.
        F.when(missing_text("customer_id"), "missing_customer_id"),  # Check the key.
        F.when(F.col("id_occurrences") > 1, "duplicate_customer_id"),  # Check duplicates after trimming.
        F.when(missing_text("first_name"), "missing_first_name"),  # Check the first name.
        F.when(missing_text("last_name"), "missing_last_name"),  # Check the last name.
        F.when(missing_text("email"), "missing_email"),  # Check the email is present.
        F.when(~missing_text("email") & ~F.col("email").contains("@"), "email_without_at"),  # Apply the basic email rule.
        F.when(missing_text("loyalty_tier"), "missing_loyalty_tier"),  # Check the tier is present.
        F.when(~missing_text("loyalty_tier") & ~F.col("loyalty_tier").isin(*allowed_tiers), "invalid_loyalty_tier"),  # Check allowed tiers.
        F.when(F.col("join_date").isNull(), "missing_join_date"),  # Check the date.
        F.when(F.col("active_flag").isNull(), "missing_active_flag"),  # Check the boolean flag.
    ),
)

accepted_count = customers_checked_df.filter(F.col("quality_issue") == "").count()  # Count rows with no issues.
rejected_count = customers_checked_df.filter(F.col("quality_issue") != "").count()  # Count rows with issues.

print("Customers ready for Silver:", accepted_count)  # Show accepted rows.
print("Customers needing rejection:", rejected_count)  # Show rejected rows.
print("All rows accounted for:", accepted_count + rejected_count == 750)  # Confirm none disappeared.\


Customers ready for Silver: 750
Customers needing rejection: 0
All rows accounted for: True


In [8]:
silver_customers_df = (  # Build the DataFrame that will become the Silver table.
    customers_checked_df
    .filter(F.col("quality_issue") == "")  # Keep only rows that passed every rule.
    .drop("id_occurrences", "quality_issue")  # Remove temporary checking columns.
)

rejected_customers_df = (  # Build a separate DataFrame for failed rows.
    customers_checked_df
    .filter(F.col("quality_issue") != "")  # Keep rows with at least one issue.
    .drop("id_occurrences")  # Keep quality_issue so we know why each row failed.
)

print("Silver candidates:", silver_customers_df.count())  # Expect 750.
print("Rejected candidates:", rejected_customers_df.count())  # Expect 0.
print("Silver columns:", silver_customers_df.columns)  # Confirm the business columns remain.

Silver candidates: 750
Rejected candidates: 0
Silver columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'loyalty_tier', 'join_date', 'active_flag']


In [9]:
bronze_count = customers_bronze_df.count()  # Count rows entering from Bronze.
silver_count = silver_customers_df.count()  # Count rows that passed the rules.
rejected_count = rejected_customers_df.count()  # Count rows that failed the rules.

if silver_count + rejected_count != bronze_count:  # Make sure every Bronze row has an outcome.
    raise ValueError("Customer rows were lost during Silver processing")  # Stop before writing.

silver_customers_path = project_root / "lakehouse" / "silver" / "data" / "customers"  # Choose the Silver Delta folder.
silver_customers_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent folder exists.

silver_customers_df.write.format("delta").mode("overwrite").save(str(silver_customers_path))  # Save a full clean snapshot.

saved_silver_df = spark.read.format("delta").load(str(silver_customers_path))  # Read the saved table back.
saved_count = saved_silver_df.count()  # Count the saved rows.

if saved_count != silver_count:  # Check that the saved count matches the accepted count.
    raise ValueError("Saved Silver customer count does not match")  # Stop if the write needs investigation.

print("Bronze customers:", bronze_count)  # Show the starting count.
print("Saved Silver customers:", saved_count)  # Show the accepted count.
print("Rejected customers:", rejected_count)  # Show rows needing separate error evidence.
print("Delta log exists:", (silver_customers_path / "_delta_log").is_dir())  # Confirm the folder is a Delta table.

Bronze customers: 750
Saved Silver customers: 750
Rejected customers: 0
Delta log exists: True
